In [1]:
import torch
from PIL import Image
from transformers import XLMRobertaTokenizer
from torchvision import transforms
from tqdm import tqdm
import pandas as pd
import os
import json
from collections import Counter
import torch.nn.functional as F

/data/2_data_server/cv-07/anaconda3/envs/beit3_env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- 0. 실행 환경 설정 ---
# 사용하려는 GPU의 ID를 설정합니다. (예: "0" 또는 "0,1")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

# --- 1. 기본 설정 및 모델 구조 import ---
# BEiT-3 레포지토리의 모델링 파일을 import해야 합니다.
# 이 파일이 있는 경로를 sys.path에 추가하거나, 같은 디렉토리에 복사해두세요.
try:
    from modeling_finetune import beit3_large_patch16_768_vqav2
except ImportError:
    print("오류: 'modeling_finetune.py'를 찾을 수 없습니다.")
    print("BEiT-3 레포지토리에서 해당 파일을 다운로드하여 이 스크립트와 같은 디렉토리에 위치시켜 주세요.")
    exit()

사용 장치: cuda


In [3]:
# --- 2. 경로 설정 (사용자 환경에 맞게 수정 필수) ---
# Fine-tuning된 모델 가중치 파일 경로
model_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3_large_indomain_patch16_768_vgqaaug_vqa.pth"
# 문장 토크나이저 모델 경로
spm_model_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3.spm"
# VQAv2 답변 라벨 파일 경로
vqa_answer_label_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/v2_mscoco_train2014_annotations.json"
# 캡션이 포함된 CSV 파일 경로 (이전 LLM 단계에서 생성한 파일)
caption_csv_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/test_with_captions.csv'
# 원본 이미지 디렉토리 경로
image_dir = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/'
# 최종 제출 파일 저장 경로 (A, B, C, D 형식)
submission_path = './submission_final.csv'
# 원본 샘플 제출 파일 경로 (ID 순서 정렬용)
sample_submission_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/sample_submission.csv'


In [4]:
# --- 3. 모델 및 토크나이저 불러오기 ---
print("\n모델과 토크나이저를 로딩합니다...")
tokenizer = XLMRobertaTokenizer(spm_model_path)
model = beit3_large_patch16_768_vqav2()

# 모델 가중치 파일 존재 여부 확인
if not os.path.exists(model_path):
    print(f"오류: 모델 가중치 파일을 찾을 수 없습니다 - {model_path}")
    exit()

checkpoint = torch.load(model_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])
model.to(device)
model.eval()
print("모델 로딩 완료.")


모델과 토크나이저를 로딩합니다...
모델 로딩 완료.


In [5]:
# --- 4. VQAv2 답변 사전 생성 ---
print("\nVQAv2 답변 사전을 생성합니다...")
try:
    with open(vqa_answer_label_path, 'r') as f:
        vqa_data = json.load(f)
    raw_annotations = vqa_data['annotations']
    answer_counter = Counter(ann['multiple_choice_answer'] for ann in raw_annotations)
    num_top_answers = 3129
    top_answers = answer_counter.most_common(num_top_answers)
    answer_vocab = [answer for answer, count in top_answers]
    answer_to_idx = {answer: i for i, answer in enumerate(answer_vocab)}

    # 'yes'와 'no'의 인덱스를 미리 찾아둡니다.
    if 'yes' in answer_to_idx and 'no' in answer_to_idx:
        yes_idx = answer_to_idx['yes']
        no_idx = answer_to_idx['no']
        print(f"답변 사전 생성 완료. 'yes' 인덱스: {yes_idx}, 'no' 인덱스: {no_idx}")
    else:
        raise ValueError("'yes' 또는 'no'를 답변 사전에서 찾을 수 없습니다!")

except (FileNotFoundError, KeyError, ValueError) as e:
    print(f"답변 사전 생성 중 오류 발생: {e}")
    exit()


VQAv2 답변 사전을 생성합니다...
답변 사전 생성 완료. 'yes' 인덱스: 0, 'no' 인덱스: 1


In [6]:
# --- 5. 데이터 로드 및 추론 준비 ---
print(f"\n캡션 데이터를 '{caption_csv_path}'에서 로드합니다.")
if not os.path.exists(caption_csv_path):
    print(f"오류: 캡션 CSV 파일을 찾을 수 없습니다 - {caption_csv_path}")
    exit()
test_df = pd.read_csv(caption_csv_path)

# 이미지 전처리(Transform) 정의
transform = transforms.Compose([
    transforms.Resize((768, 768)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

# --- 6. 캡션을 이용한 추론 실행 ---
print("\n캡션을 'Yes or No' 질문으로 변환하여 추론을 시작합니다...")
results = []
pbar = tqdm(test_df.iterrows(), total=len(test_df))

for _, row in pbar:
    image_path = os.path.join(image_dir, row['img_path'].lstrip('./'))

    if not os.path.exists(image_path):
        print(f"Warning: 이미지를 찾을 수 없습니다 - {image_path}")
        results.append({'ID': row['ID'], 'answer_text': '?'})
        continue
    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)
    except Exception as e:
        print(f"이미지 처리 오류 {image_path}: {e}")
        results.append({'ID': row['ID'], 'answer_text': '?'})
        continue

    choice_scores = {}
    for choice_key in ['A', 'B', 'C', 'D']:
        caption = row[f'caption_{choice_key}']
        if not isinstance(caption, str) or not caption:
            choice_scores[choice_key] = -1.0
            continue

        question_text = f"{caption}. Yes or no?"
        pbar.set_description(f"Processing ID: {row['ID']}, Choice: {choice_key}")

        encoding = tokenizer(
            question_text, padding='max_length', truncation=True,
            max_length=40, return_tensors='pt'
        )
        question_tokens = encoding['input_ids'].to(device)
        padding_mask = encoding['attention_mask'].to(device)

        with torch.no_grad():
            output_logits = model(
                image=image_tensor,
                question=question_tokens,
                padding_mask=padding_mask
            )

        yes_logit = output_logits[0, yes_idx].item()
        no_logit = output_logits[0, no_idx].item()
        
        logits_tensor = torch.tensor([no_logit, yes_logit])
        probabilities = F.softmax(logits_tensor, dim=0)
        yes_prob = probabilities[1].item()
        
        choice_scores[choice_key] = yes_prob

    if choice_scores:
        best_choice_key = max(choice_scores, key=choice_scores.get)
        predicted_answer_text = row[best_choice_key]
    else:
        predicted_answer_text = row['A']

    results.append({'ID': row['ID'], 'answer_text': predicted_answer_text})

print("\n추론 완료! 최종 제출 파일 생성을 시작합니다.")


캡션 데이터를 '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/test_with_captions.csv'에서 로드합니다.

캡션을 'Yes or No' 질문으로 변환하여 추론을 시작합니다...


Processing ID: TEST_679, Choice: D: 100%|██████████| 680/680 [05:17<00:00,  2.14it/s]


추론 완료! 최종 제출 파일 생성을 시작합니다.


In [8]:
# --- 7. 최종 결과 제출 파일로 저장 (내용 -> A/B/C/D 변환) ---
text_submission_df = pd.DataFrame(results)
merged_df = pd.merge(text_submission_df, test_df[['ID', 'A', 'B', 'C', 'D']], on='ID', how='left')

def convert_text_to_choice(row):
    predicted_text = row['answer_text']
    for choice in ['A', 'B', 'C', 'D']:
        if predicted_text == row[choice]:
            return choice
    # 일치하는 보기가 없을 경우 (추론 오류 또는 ? 처리)
    print(f"Warning: ID {row['ID']}의 예측 답변 '{predicted_text}'가 보기와 일치하지 않습니다. 기본값 'A'로 처리합니다.")
    return 'A'

merged_df['answer'] = merged_df.apply(convert_text_to_choice, axis=1)
final_submission_df = merged_df[['ID', 'answer']]

# 원본 제출 파일 형식에 맞게 ID 순으로 정렬
try:
    sample_df = pd.read_csv(sample_submission_path)
    final_submission_df = final_submission_df.set_index('ID').loc[sample_df['ID']].reset_index()
    print("\n샘플 제출 파일에 맞게 ID 순서 정렬 완료.")
except Exception as e:
    print(f"\nWarning: ID 순서 정렬 중 오류 발생: {e}. 정렬을 건너뜁니다.")


# 최종 결과 저장
output_dir = os.path.dirname(submission_path)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir)

final_submission_df.to_csv(submission_path, index=False)

print(f"\n✅ 모든 작업 완료! 최종 제출 파일이 '{submission_path}'에 저장되었습니다.")
print("\n--- 최종 제출 파일 미리보기 (상위 5개) ---")
print(final_submission_df.head())


샘플 제출 파일에 맞게 ID 순서 정렬 완료.

✅ 모든 작업 완료! 최종 제출 파일이 './submission_final.csv'에 저장되었습니다.

--- 최종 제출 파일 미리보기 (상위 5개) ---
         ID answer
0  TEST_000      A
1  TEST_001      C
2  TEST_002      C
3  TEST_003      B
4  TEST_004      C
